# R7.7 | Reviewer experiment 4 — model complexity, efficiency, input noise robustness & multi-seed stability

**论文模型名称只用 `hybrid deep-learning prediction`。** Internal architecture (reproducibility only): R6.5 GN8 / CrossDrop60. This notebook does **not** change model architecture, hyperparameters or comparison years. Input noise is placed **only in the historical 180-day path**, before frozen train-only EOF, while ground truth remains unmodified. Tests remain the causal rolling annual folds of R7.5 (10 years, 30/60/90-day leads).

**Note:** 2011/2014/2019 were exposed in development. This is a robustness/computational evaluation of a fixed protocol, not a new independent blind benchmark. Reused 2014/2019 checkpoints have zero *new* training time and must **not** be treated as zero-cost training.

**Execution:** Run All; preflight automatically runs before formal seed training. New seeds 73–75 trained for *every* fold, alongside frozen R7.5 seeds 71,72. Reports, progress, best/latest weights, inference and per-issue score files are checkpointed per fold/seed. No retraining of R7.5 seed 71/72.

In [ ]:
# 0. Freeze R7.5 protocol. Configure ROOT only if D: is mounted elsewhere.
from pathlib import Path
from dataclasses import dataclass,asdict
from datetime import datetime,timezone
import os,sys,json,hashlib,math,time,random,gc,traceback,zipfile,warnings,re,copy,platform
import numpy as np,pandas as pd
from scipy.spatial import cKDTree
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
import matplotlib.pyplot as plt
from IPython.display import display,FileLink
from tqdm.auto import tqdm
PAPER_NAME='hybrid deep-learning prediction';INTERNAL_ARCH='R65-CrossDrop60'
@dataclass
class Settings:
    ROOT:str='/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL:str='final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    OUTPUT_REL:str='R7_7_FrozenHybrid_Complexity_Noise_MultiSeed'
    R75_REL:str='R7_5_HybridDeepLearning_MultiYearRollingHindcast'
    R76_REL:str='R7_6_FairBaselines_2026'
    SOURCE_R65_WEIGHTS_REL:str='R6_5_跨分支交换随机屏蔽实验/模型权重/R65_Checkpoints'
    HISTORY:int=180
    LEADS:tuple=(30,60,90)
    TEST_YEARS:tuple=(2005,2008,2011,2014,2017,2019,2021,2022,2023,2024)
    SEEDS:tuple=(71,72,73,74,75)
    R75_SEEDS:tuple=(71,72)
    NEW_SEEDS:tuple=(73,74,75)
    PURGE_DAYS:int=7
    MAX_INPUT_GAP:int=2
    MAX_MISSING_FRAC:float=.03
    DROP_BRANCH1_FIRST:int=12
    BRANCH2_MAX_LON:float=145.0
    EOF_VARIANCE_TARGET:float=.99
    BATCH_SIZE:int=32
    DROPOUT:float=.30
    CROSS_DROPOUT:float=.60
    ALPHA:float=.40
    KM_SCALE:float=25.0
    HUBER_BETA:float=.50
    LR:float=.001
    WEIGHT_DECAY:float=.0001
    EPOCHS:int=36
    PATIENCE:int=8
    USE_AMP:bool=True
    FORCE_TRAIN:bool=False
    USE_EXISTING_R65_WEIGHTS:bool=False
    RUN_FULL:bool=True
    ONLY_PREFLIGHT:bool=False
    MIN_TRAIN:int=500
    MIN_VAL:int=100
    MIN_TEST:int=100
    NUM_WORKERS:int=0
    NOISE_LEVELS_KM:tuple=(0.0,5.0,10.0,20.0)
    NOISE_REPEATS:int=3
    NOISE_SEED:int=77119
    BENCH_BATCH_SIZE:int=32
    BENCH_WARMUP:int=5
    BENCH_REPEAT:int=25
    BOOT_REPS:int=800
    INCLUDE_WEIGHTS_IN_ZIP:bool=False
    FIG_DPI:int=220
c=Settings()
ROOT=Path(c.ROOT);DS=ROOT/c.DATASET_REL
RUN=ROOT/c.OUTPUT_REL
TABLES=RUN/'tables';WEIGHTS=RUN/'new_seed_weights';FORECAST=RUN/'noise_predictions_cache';LOGS=RUN/'logs';FIG=RUN/'figures_en';REPORTS=RUN/'reports';EOFSTORE=RUN/'trainonly_eof'
R75=ROOT/c.R75_REL;R76=ROOT/c.R76_REL
R75_WEIGHTS=R75/'weights';R75_FORECAST=R75/'forecast_arrays';R75_EOF=R75/'trainonly_eof';R76_WEIGHTS=R76/'weights'
for q in [RUN,TABLES,WEIGHTS,FORECAST,LOGS,FIG,REPORTS,EOFSTORE]:q.mkdir(parents=True,exist_ok=True)
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not DS.is_file():raise FileNotFoundError('冻结数据集不存在: '+str(DS))
if not R75_WEIGHTS.is_dir() or not R75_FORECAST.is_dir():raise FileNotFoundError('缺少R7.5本地权重或forecast_arrays，请先恢复R7.5完整运行目录')
if not R76_WEIGHTS.is_dir():raise FileNotFoundError('缺少R7.6本地LSTM weights，无法做同设备效率比较')
if set(c.NEW_SEEDS)&set(c.R75_SEEDS):raise RuntimeError('新/旧随机种子重叠')
if len(c.NOISE_LEVELS_KM)!=len(set(c.NOISE_LEVELS_KM)) or c.NOISE_LEVELS_KM[0]!=0:raise RuntimeError('噪声合同不合法')
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8<<20),b''):h.update(b)
    return h.hexdigest()
DATASET_SHA=sha256_file(DS)
print('Model',PAPER_NAME,'device',DEV,'torch',torch.__version__,'dataset SHA',DATASET_SHA)
print('R75:',R75,'R76:',R76,'R77:',RUN)
(REPORTS/'R77_contract.json').write_text(json.dumps({'model':PAPER_NAME,'internal_architecture_for_audit':INTERNAL_ARCH,'dataset_sha256':DATASET_SHA,'settings':asdict(c),'development_exposed_years':[2011,2014,2019],'noise_is_input_only':'yes','architecture_frozen':'yes'},ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:
# 1｜按冻结数据格式恢复双分支/日历，不修改样本或标签
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
if dates.has_duplicates or not(len(dates)==len(b1)==len(b2)):
    raise RuntimeError('冻结数据重复日期或数量不一致')
order=np.argsort(dates.values);dates=dates[order];b1=b1[order];b2=b2[order]
if not(np.isfinite(b1).all() and np.isfinite(b2).all()):raise RuntimeError('输入冻结数据含NaN/Inf')
b1=b1[:,c.DROP_BRANCH1_FIRST:];lat1=lat1[c.DROP_BRANCH1_FIRST:]
keep2=lon2<=c.BRANCH2_MAX_LON+1e-8;lon2=lon2[keep2];b2=b2[:,keep2]
cal=pd.date_range(dates.min(),dates.max(),freq='D');pos=cal.get_indexer(dates)
if (pos<0).any():raise RuntimeError('时间轴不合法')
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32)
A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1;A2[pos]=b2
# 缺失值仅以前向可用历史补齐；未来标签永远取 A1/A2 的真实日样本
F1=pd.DataFrame(A1).ffill(limit=c.MAX_INPUT_GAP).to_numpy(dtype=np.float32)
F2=pd.DataFrame(A2).ffill(limit=c.MAX_INPUT_GAP).to_numpy(dtype=np.float32)
H=int(c.HISTORY); L=max(c.LEADS)
missing=(~valid).astype(np.int32);iss=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]:continue
    win=missing[i-H+1:i+1]
    if win.mean()>c.MAX_MISSING_FRAC:continue
    if np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>c.MAX_INPUT_GAP:continue
    if any(not valid[i+d] for d in c.LEADS):continue
    if not(np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()):continue
    iss.append(i)
ISS=np.asarray(iss,np.int32)
if not len(ISS):raise RuntimeError('没有可用180→30/60/90起报日期')
if not all(np.all((cal[ISS+d]-cal[ISS]).days==d) for d in c.LEADS):raise RuntimeError('日期错位')
print('Frozen dates',len(dates),'range',dates.min().date(),dates.max().date(),'valid anchors:',len(ISS),'B1dim',b1.shape[1],'B2dim',b2.shape[1])


In [ ]:
# 2｜各年的扩张窗口：训练标签严格早于上一年验证起报，验证标签早于测试起报
FOLDS=[];fr=[];ar=[]
for y in c.TEST_YEARS:
    val_start=pd.Timestamp(int(y)-1,1,1)
    test_start=pd.Timestamp(int(y),1,1)
    next_start=pd.Timestamp(int(y)+1,1,1)
    gap=pd.Timedelta(days=c.PURGE_DAYS)
    train_cut=val_start-gap
    train=ISS[cal[ISS+L]<train_cut]
    val=ISS[(cal[ISS]>=val_start)&(cal[ISS+L]<test_start-gap)]
    test=ISS[(cal[ISS]>=test_start)&(cal[ISS+L]<next_start-gap)]
    if len(train)<c.MIN_TRAIN or len(val)<c.MIN_VAL or len(test)<c.MIN_TEST:
        raise RuntimeError(f'{y} 时序样本不足 train/val/test={(len(train),len(val),len(test))}')
    assert cal[train+L].max() < cal[val].min()-gap
    assert cal[val+L].max() < cal[test].min()-gap
    if not set(cal[train+L]).isdisjoint(cal[val+L]) or not set(cal[val+L]).isdisjoint(cal[test+L]):
        raise RuntimeError(f'{y} 目标日期泄露/相交')
    f={'year':int(y),'train':train,'val':val,'test':test,'train_cut':train_cut,'val_start':val_start,'test_start':test_start}
    FOLDS.append(f)
    fr.append({'test_year':y,'train_n':len(train),'validation_n':len(val),'test_n':len(test),
               'train_last_issue':str(cal[train[-1]].date()),'train_last_target_90d':str(cal[train[-1]+L].date()),
               'validation_first_issue':str(cal[val[0]].date()),'validation_last_target_90d':str(cal[val[-1]+L].date()),
               'test_first_issue':str(cal[test[0]].date()),'test_last_target_90d':str(cal[test[-1]+L].date()),
               'gap_days':c.PURGE_DAYS,'strict_causal':'PASS','seen_in_development':int(y) in (2014,2019)})
    for split,ii in [('train',train),('validation',val),('test',test)]:
        ar.extend({'test_year':y,'split':split,'issue_date':str(cal[i].date()),'target_30':str(cal[i+30].date()),'target_60':str(cal[i+60].date()),'target_90':str(cal[i+90].date())} for i in ii)
fold_table=pd.DataFrame(fr);fold_table.to_csv(TABLES/'01_fold_dates_causal_audit.csv',index=False)
pd.DataFrame(ar).to_csv(TABLES/'02_all_fold_issue_dates_for_future_baselines.csv',index=False)
print('Folds STRICT CAUSAL PASS');display(fold_table)


In [ ]:
# 3｜每折独立在训练日期拟合标准化与99% EOF。训练/验证/测试共享该折拟合结果。
class DailyEOF:
    def fit(self,obs,target=.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0);self.sd=np.std(obs,axis=0);self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,V=np.linalg.eigh(cov);ii=np.argsort(eig)[::-1]
        eig=np.maximum(eig[ii],0);V=V[:,ii]
        ratios=np.cumsum(eig)/(eig.sum()+1e-12);k=int(np.searchsorted(ratios,target)+1)
        self.V=np.asarray(V[:,:k],np.float32);self.explained=float(ratios[k-1]);self.n=k
        return self
    def z(self,x):return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)
    def pc(self,x):return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,x):return ((np.asarray(x)@self.V.T)*self.sd+self.mu).astype(np.float32)

def make_fold_data(f):
    allowed=(cal<f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[allowed],c.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[allowed],c.EOF_VARIANCE_TARGET)
    if min(pp1.explained,pp2.explained)<c.EOF_VARIANCE_TARGET-1e-6:raise RuntimeError('train-only EOF方差未达标')
    fd=dict(pp1=pp1,pp2=pp2,z1=pp1.z(F1),z2=pp2.z(F2),p1=pp1.pc(F1),p2=pp2.pc(F2))
    snap=EOFSTORE/f"trainonly_EOF_{f['year']}.npz"
    np.savez_compressed(snap,b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
                        lat1=lat1,lon2=lon2,train_cut=str(f['train_cut'].date()),dataset_sha256=DATASET_SHA)
    return fd


In [ ]:
# 3 原架构代码：不更改 GroupNorm 分组数、Dropout、输入特征或层名。
import torch.nn.functional as F
class ChannelSeparatedEncoder(nn.Module):
    """单时刻空间卷积；输入[B,T,2*M] => Conv1d [B*T,2,M]."""
    def __init__(self,feature,branch):
        super().__init__()
        assert feature%2==0, '需要[x, delta]等长拼接输入'
        self.raw_dim=feature//2
        base_c=16 if branch==1 else 32
        k=3 if branch==1 else 5
        channels=base_c*2
        # 正确的对称空间填充：dilation=2 -> padding=k-1，输出仍为M
        self.conv=nn.Conv1d(2,channels,kernel_size=k,dilation=2,padding=k-1)
        self.bn=nn.BatchNorm1d(channels)
        self.fc=nn.Linear(channels*self.raw_dim,32 if branch==1 else 64)
        self.out_dim=self.fc.out_features
        self.drop=nn.Dropout(c.DROPOUT)
    def channelize(self,x):
        B,T,F=x.shape
        if F!=self.raw_dim*2: raise RuntimeError('R53特征维不匹配')
        return torch.stack((x[:,:,:self.raw_dim],x[:,:,self.raw_dim:]),dim=2).reshape(B*T,2,self.raw_dim)
    def forward(self,x):
        B,T,F=x.shape
        z=self.channelize(x)
        z=self.drop(torch.relu(self.bn(self.conv(z))))
        if z.shape[2]!=self.raw_dim:raise RuntimeError('R53 spatial-conv输出宽度发生变化')
        z=self.drop(torch.relu(self.fc(z.flatten(1))))
        return z.reshape(B,T,-1)
class ChannelHybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,projection=None,use_pc_skip=False):
        super().__init__()
        self.enc=ChannelSeparatedEncoder(feature,branch)
        self.use_pc_skip=bool(use_pc_skip)
        self.raw_dim=feature//2
        self.pc_dim=pc_dim
        if self.use_pc_skip:
            self.register_buffer('V',torch.as_tensor(projection,dtype=torch.float32))
        hidden=32 if branch==1 else 64
        in_dim=self.enc.out_dim+(2*pc_dim if self.use_pc_skip else 0)
        self.rnn=nn.LSTM(in_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
    def eof_skip(self,x):
        if not self.use_pc_skip: raise RuntimeError('no PC skip in this model')
        return torch.cat((x[:,:,:self.raw_dim]@self.V,x[:,:,self.raw_dim:]@self.V),dim=-1)
    def forward(self,x):
        z=self.enc(x)
        if self.use_pc_skip: z=torch.cat((z,self.eof_skip(x)),dim=-1)
        h,_=self.rnn(z)
        return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def make_r53_model(name,branch,fd):
    if name!='R53-ChannelConv-PCConcat':raise ValueError('只允许原R53-PCConcat编码器')
    feature=fd[f'z{branch}'].shape[1]*2; pdim=fd[f'pp{branch}'].n
    return ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
class JointHiddenEOF(nn.Module):
    def __init__(self,fd,mode='cross'):
        super().__init__()
        if mode not in ('cross','self'): raise ValueError(mode)
        self.mode=mode
        self.net1=make_r53_model('R53-ChannelConv-PCConcat',1,fd)
        self.net2=make_r53_model('R53-ChannelConv-PCConcat',2,fd)
        d1=int(self.net1.fc.in_features);d2=int(self.net2.fc.in_features)
        self.exchange1=nn.Linear(d2 if mode=='cross' else d1,d1)
        self.exchange2=nn.Linear(d1 if mode=='cross' else d2,d2)
        nn.init.zeros_(self.exchange1.weight);nn.init.zeros_(self.exchange1.bias)
        nn.init.zeros_(self.exchange2.weight);nn.init.zeros_(self.exchange2.bias)
    def _hidden(self,net,x):
        a=net.enc(x)
        a=torch.cat([a,net.eof_skip(x)],dim=-1)
        h,_=net.rnn(a)
        return h[:,-1]
    def forward(self,x1,x2):
        h1=self._hidden(self.net1,x1)
        h2=self._hidden(self.net2,x2)
        if self.mode=='cross':
            t1=h1+self.exchange1(torch.tanh(h2))
            t2=h2+self.exchange2(torch.tanh(h1))
        else:
            t1=h1+self.exchange1(torch.tanh(h1))
            t2=h2+self.exchange2(torch.tanh(h2))
        o1=self.net1.fc(t1).reshape(len(x1),3,self.net1.pc_dim)
        o2=self.net2.fc(t2).reshape(len(x2),3,self.net2.pc_dim)
        return o1,o2
class CrossExchangeDropGN(JointHiddenEOF):
    def __init__(self,fd,drop_p):
        super().__init__(fd,'cross')
        self.drop_p=float(drop_p)
        assert 0 <= self.drop_p < 1.0
        for net in (self.net1,self.net2):
            ch=net.enc.conv.out_channels
            assert ch%8==0
            net.enc.bn=nn.GroupNorm(8,ch)

    def forward(self,x1,x2):
        h1=self._hidden(self.net1,x1)
        h2=self._hidden(self.net2,x2)
        m1=self.exchange1(torch.tanh(h2))
        m2=self.exchange2(torch.tanh(h1))
        # 训练期对隐藏特征做独立 dropout；推理时恒等，严禁随机测试增强
        m1=F.dropout(m1,p=self.drop_p,training=self.training)
        m2=F.dropout(m2,p=self.drop_p,training=self.training)
        y1=self.net1.fc(h1+m1).reshape(len(x1),3,self.net1.pc_dim)
        y2=self.net2.fc(h2+m2).reshape(len(x2),3,self.net2.pc_dim)
        return y1,y2

def r65_model(fd,name):
    if name!='R65-CrossDrop60':raise ValueError('仅允许冻结的单模型架构')
    return CrossExchangeDropGN(fd,0.60)

In [ ]:
# 5｜R6.5 原版联合双分支 Dataset + R6.1 物理距离损失；所有参数固定
class JointPairDataset(Dataset):
    def __init__(self,issues,fd):
        self.issues=np.asarray(issues,np.int64)
        self.z1=fd['z1'];self.z2=fd['z2'];self.p1=fd['p1'];self.p2=fd['p2']
    def __len__(self):return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j])
        def pack(z):
            v=z[i-H+1:i+1]
            d=np.empty_like(v);d[0]=0;d[1:]=v[1:]-v[:-1]
            return torch.from_numpy(np.concatenate([v,d],axis=-1).copy())
        y1=np.stack([self.p1[i+d] for d in c.LEADS]);y2=np.stack([self.p2[i+d] for d in c.LEADS])
        return pack(self.z1),pack(self.z2),torch.from_numpy(y1.copy()),torch.from_numpy(y2.copy())

def r61_loss(pred,target,fd,branch):
    # exact original R61 differentiable km-backprojection loss
    pc_mse=F.mse_loss(pred.float(),target.float())
    pp=fd[f'pp{branch}']
    V=torch.as_tensor(pp.V,dtype=torch.float32,device=pred.device)
    sd=torch.as_tensor(pp.sd,dtype=torch.float32,device=pred.device)
    scale=(torch.as_tensor(111.32*np.cos(np.deg2rad(lat1)),dtype=torch.float32,device=pred.device)
           if branch==1 else torch.ones_like(sd)*111.32)
    km=((pred.float()-target.float())@V.T)*((sd*scale)[None,None,:])
    phy=F.smooth_l1_loss(km/c.KM_SCALE,torch.zeros_like(km),beta=c.HUBER_BETA)
    return (1-c.ALPHA)*pc_mse+c.ALPHA*phy

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def nparams(m):return sum(int(p.numel()) for p in m.parameters())


In [ ]:
# 6｜后报评分：保留 R6.5 双向最近线段距离定义，同时报告日期及双分支物理坐标误差
# 该指标用于 GLORYS 50m 派生路径后报；不能与 ADT 参考误差直接视为同一真值。
def merge_path(b1coord,b2coord):
    a=np.asarray(b1coord,float);b=np.asarray(b2coord,float)
    candidates=np.flatnonzero(lon2>a[-1])
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    x=np.r_[a,lon2[k:]];y=np.r_[lat1,b[k:]]
    mask=np.isfinite(x)&np.isfinite(y)
    return x[mask],y[mask]

def one_direction(A,B):
    if min(len(A),len(B))<2:return np.asarray([np.nan])
    tree=cKDTree(B);_,kidx=tree.query(A,k=min(3,len(B)))
    kidx=np.asarray(kidx)
    if kidx.ndim==1:kidx=kidx[:,None]
    segments=np.unique(np.clip(np.concatenate([kidx.ravel()-1,kidx.ravel()]),0,len(B)-2))
    v=B[segments+1]-B[segments]
    t=np.clip(np.sum((A[:,None,:]-B[segments][None,:,:])*v[None,:,:],axis=2)/np.maximum(np.sum(v*v,axis=1),1e-12)[None,:],0,1)
    closest=B[segments][None,:,:]+t[:,:,None]*v[None,:,:]
    return np.sqrt(((A[:,None,:]-closest)**2).sum(axis=-1)).min(axis=1)

def errors_for_pair(p1,p2,y1,y2):
    px,py=merge_path(p1,p2);tx,ty=merge_path(y1,y2)
    lat0=float(np.mean(np.r_[py,ty]));lon0=float(np.mean(np.r_[px,tx]))
    cosine=np.cos(np.deg2rad(lat0))
    P=np.column_stack(((px-lon0)*111.32*cosine,(py-lat0)*111.32))
    T=np.column_stack(((tx-lon0)*111.32*cosine,(ty-lat0)*111.32))
    d=np.r_[one_direction(P,T),one_direction(T,P)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def score_predictions(f,seed,b1pred,b2pred):
    out=[]
    for j,i in enumerate(f['test']):
        for k,d in enumerate(c.LEADS):
            mse,mae,p90=errors_for_pair(b1pred[j,k],b2pred[j,k],A1[i+d],A2[i+d])
            if not np.isfinite(mse):raise RuntimeError('非有限预测路径评分')
            diff1=(b1pred[j,k]-A1[i+d])*111.32*np.cos(np.deg2rad(lat1))
            diff2=(b2pred[j,k]-A2[i+d])*111.32
            out.append(dict(test_year=int(f['year']),seed=int(seed),model=PAPER_NAME,issue_date=str(cal[i].date()),
                            target_date=str(cal[i+d].date()),lead_days=int(d),
                            path_mse_km2=mse,path_rmse_km=math.sqrt(mse),path_mae_km=mae,path_p90_km=p90,
                            branch1_km_rmse=float(np.sqrt(np.mean(diff1**2))),branch2_km_rmse=float(np.sqrt(np.mean(diff2**2))),
                            previously_development_year=int(f['year']) in (2014,2019)))
    return pd.DataFrame(out)


### Frozen model contracts & seed training code
This reuses the **exact** R7.5 model training, early-stopping and fold-specific CID rules. Only seeds 73–75 are trained into a **separate** R7.7 directory. Older seeds 71/72 are restored exclusively from R7.5 weights. No target-year score is used to set hyperparameters.

In [ ]:
# 8｜严格合同的训练/早停/恢复；2014/2019可只在CID签名一致时复用旧冻结权重
# checkpoints 每 epoch 保存 latest.pt；Best永不依据测试集成绩选择。
ARCHITECTURE_ID=hashlib.sha256((INTERNAL_ARCH+'|GN8|crossdrop0.60|pcskip|physical-alpha0.4').encode()).hexdigest()
LEGACY_SEEDS=(71,72)
LEGACY_DEV_YEARS=(2014,2019)
LEGACY_CONFIRM_YEARS=(2006,2011)
LEGACY_PROTOCOL=dict(
    version='R65_GN8_CrossExchangeDrop_v1',dataset_sha256=DATASET_SHA,data_representation='Original frozen',
    history_days=c.HISTORY,lead_days=list(c.LEADS),physical_loss_alpha=.40,
    development_years=list(LEGACY_DEV_YEARS),confirmation_development_years=list(LEGACY_CONFIRM_YEARS),
    seeds=list(LEGACY_SEEDS),epochs=36,patience=8,dropout=.30,cross_exchange_dropout=[.35,.60],
    lr=.001,weight_decay=.0001,reference='same-data same-split R6.4 genuine prediction archive',
    model_choice_from_confirm_years=False,all_QA_years_development_not_blind=True)

def legacy_cid(f,fd,seed):
    old={**LEGACY_PROTOCOL,'QA_year':int(f['year']),'seed':int(seed),'name':INTERNAL_ARCH,
         'tr_end':int(f['train'][-1]),'va_end':int(f['val'][-1]),
         'nPC1':int(fd['pp1'].n),'nPC2':int(fd['pp2'].n)}
    return hashlib.sha256(json.dumps(old,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def new_contract(f,fd,seed):
    return dict(protocol='R75_expanding_hindcast_v1',arch=ARCHITECTURE_ID,dataset_sha256=DATASET_SHA,
                model_name=PAPER_NAME,year=f['year'],seed=seed,train_end=int(f['train'][-1]),
                val_end=int(f['val'][-1]),test_first=str(cal[f['test'][0]].date()),
                split_gap=c.PURGE_DAYS,eof1=fd['pp1'].n,eof2=fd['pp2'].n,
                epochs=c.EPOCHS,patience=c.PATIENCE,batch=c.BATCH_SIZE,lr=c.LR,wd=c.WEIGHT_DECAY,
                loss_alpha=c.ALPHA,km_scale=c.KM_SCALE,huber=c.HUBER_BETA,history=c.HISTORY,leads=list(c.LEADS),
                p_cross=c.CROSS_DROPOUT,p_encoder=c.DROPOUT,amp=c.USE_AMP,train_only_eof=True)

def cid_from_contract(contract):return hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def restore_completed_or_legacy(f,fd,seed,model,cid,folder):
    best=folder/'best.pt'; status=folder/'status.json'
    if not c.FORCE_TRAIN and best.is_file() and status.is_file():
        ck=torch.load(best,map_location='cpu',weights_only=True)
        if ck.get('cid')!=cid:raise RuntimeError('已存在但合同不一致的 best.pt：'+str(best))
        model.load_state_dict(ck['state'],strict=True)
        meta=json.loads(status.read_text(encoding='utf-8'))
        if meta.get('cid')!=cid:raise RuntimeError('status与best CID不一致')
        print('[R75 REUSE]',f['year'],seed,'frozen fold checkpoint')
        return model,meta
    if c.USE_EXISTING_R65_WEIGHTS and f['year'] in LEGACY_DEV_YEARS and seed in LEGACY_SEEDS and not c.FORCE_TRAIN:
        pt=ROOT/c.SOURCE_R65_WEIGHTS_REL/str(f['year'])/str(seed)/INTERNAL_ARCH/'best.pt'
        if pt.is_file():
            old=torch.load(pt,map_location='cpu',weights_only=True)
            expected=legacy_cid(f,fd,seed)
            if old.get('cid')==expected:
                model.load_state_dict(old['state'],strict=True)
                folder.mkdir(exist_ok=True,parents=True)
                meta=dict(year=f['year'],seed=seed,cid=cid,source='exact_legacy_R65_CID',legacy_checkpoint_sha=sha256_file(pt),
                          best_epoch=int(old.get('epoch',0)),best_val=float(old.get('best_val',np.nan)),elapsed_s=0.0,
                          model_parameters=nparams(model),arch=INTERNAL_ARCH)
                torch.save({'cid':cid,'state':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                            'epoch':meta['best_epoch'],'best_val':meta['best_val'],'legacy_cid':expected},best)
                status.write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
                print('[R75 EXACT LEGACY REUSE]',f['year'],seed,'sha',meta['legacy_checkpoint_sha'][:12])
                return model,meta
            print('[R75 LEGACY MISMATCH: training from scratch]',f['year'],seed, str(pt))
    return None,None

def run_train(f,fd,seed):
    seed_all(seed)
    m=r65_model(fd,INTERNAL_ARCH).to(DEV)
    folder=WEIGHTS/str(f['year'])/str(seed);folder.mkdir(exist_ok=True,parents=True)
    cid=cid_from_contract(new_contract(f,fd,seed))
    reused,meta=restore_completed_or_legacy(f,fd,seed,m,cid,folder)
    if reused is not None:return reused,meta
    tr=DataLoader(JointPairDataset(f['train'],fd),batch_size=c.BATCH_SIZE,shuffle=True,num_workers=c.NUM_WORKERS)
    va=DataLoader(JointPairDataset(f['val'],fd),batch_size=c.BATCH_SIZE,shuffle=False,num_workers=c.NUM_WORKERS)
    opt=torch.optim.AdamW(m.parameters(),lr=c.LR,weight_decay=c.WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,patience=4,factor=.5)
    amp=DEV.type=='cuda' and bool(c.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    first=1;best=float('inf');bad=0;history=[];prior_elapsed=0.0
    latest=folder/'latest.pt';bestfile=folder/'best.pt';status=folder/'status.json'
    if latest.exists() and not c.FORCE_TRAIN:
        state=torch.load(latest,map_location='cpu',weights_only=True)
        if state.get('cid')!=cid:raise RuntimeError('latest.pt 与新协议不一致，需要另选输出目录')
        m.load_state_dict(state['state'],strict=True);opt.load_state_dict(state['opt']);sch.load_state_dict(state['sch'])
        scaler.load_state_dict(state['scaler']);first=int(state['epoch'])+1
        best=float(state['best']);bad=int(state['bad']);history=list(state['history'])
        prior_elapsed=float(state.get('elapsed_s',0.0))
        if 'rng_torch' in state:torch.set_rng_state(state['rng_torch'])
        if DEV.type=='cuda' and 'rng_cuda' in state:torch.cuda.set_rng_state_all(state['rng_cuda'])
        print('[R75 RESUME]',f['year'],seed,'epoch',first)
    epoch_started=time.perf_counter()
    for epoch in range(first,c.EPOCHS+1):
        m.train();tr_sum=0.;tr_n=0
        for x1,x2,y1,y2 in tr:
            x1,x2,y1,y2=[t.to(DEV,non_blocking=True) for t in (x1,x2,y1,y2)]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda' if amp else 'cpu',enabled=amp):
                a,b=m(x1,x2)
                loss=.5*(r61_loss(a.float(),y1.float(),fd,1)+r61_loss(b.float(),y2.float(),fd,2))
            if not torch.isfinite(loss):raise RuntimeError('训练 loss NaN/Inf')
            scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(m.parameters(),1.0)
            scaler.step(opt);scaler.update()
            tr_sum+=float(loss.detach())*len(x1);tr_n+=len(x1)
        m.eval();val_sum=0.;val_n=0
        with torch.inference_mode():
            for x1,x2,y1,y2 in va:
                x1,x2,y1,y2=[t.to(DEV,non_blocking=True) for t in (x1,x2,y1,y2)]
                with torch.autocast(device_type='cuda' if amp else 'cpu',enabled=amp):
                    a,b=m(x1,x2)
                    vloss=.5*(r61_loss(a.float(),y1.float(),fd,1)+r61_loss(b.float(),y2.float(),fd,2))
                val_sum+=float(vloss)*len(x1);val_n+=len(x1)
        value=val_sum/max(1,val_n);sch.step(value)
        if value<best-1e-7:
            best=value;bad=0
            torch.save({'cid':cid,'state':{k:v.detach().cpu() for k,v in m.state_dict().items()},'epoch':epoch,'best_val':best},bestfile)
        else:bad+=1
        history.append(dict(year=f['year'],seed=seed,epoch=epoch,train_loss=tr_sum/max(1,tr_n),val_loss=value,
                            best_val=best,lr=float(opt.param_groups[0]['lr']),bad_epochs=bad,
                            elapsed_s=prior_elapsed+time.perf_counter()-epoch_started))
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
        torch.save({'cid':cid,'state':{k:v.detach().cpu() for k,v in m.state_dict().items()},
                    'opt':opt.state_dict(),'sch':sch.state_dict(),'scaler':scaler.state_dict(),
                    'epoch':epoch,'best':best,'bad':bad,'history':history,
                    'elapsed_s':prior_elapsed+time.perf_counter()-epoch_started,
                    'rng_torch':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all() if DEV.type=='cuda' else []},latest)
        if epoch==1 or epoch%5==0 or bad>=c.PATIENCE:
            print(f'[R75 TRAIN] {f["year"]} seed {seed} epoch={epoch} train={tr_sum/max(1,tr_n):.5f} val={value:.5f} bad={bad}')
        if bad>=c.PATIENCE:break
    if not bestfile.is_file():raise RuntimeError('无最佳权重，禁止测试')
    ck=torch.load(bestfile,map_location='cpu',weights_only=True)
    if ck['cid']!=cid:raise RuntimeError('best CID 不一致')
    m.load_state_dict(ck['state'],strict=True)
    meta=dict(year=f['year'],seed=seed,cid=cid,source='new_rolling_training',
              best_epoch=int(ck['epoch']),best_val=float(ck['best_val']),model_parameters=nparams(m),
              elapsed_s=prior_elapsed+time.perf_counter()-epoch_started,arch=INTERNAL_ARCH)
    status.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
    m.eval();return m,meta

def predict_pair(model,fd,issues):
    ld=DataLoader(JointPairDataset(issues,fd),batch_size=c.BATCH_SIZE,shuffle=False,num_workers=c.NUM_WORKERS)
    a=[];b=[];model.eval()
    with torch.inference_mode():
        for x1,x2,_,_ in ld:
            p,q=model(x1.to(DEV),x2.to(DEV))
            a.append(p.float().cpu().numpy());b.append(q.float().cpu().numpy())
    return fd['pp1'].invert(np.concatenate(a)),fd['pp2'].invert(np.concatenate(b))


In [ ]:
# Complexity benchmark companion architecture from R7.6 (not retrained/selected)
# 8. 同规模LSTM：不使用TCN/Conv/跨分支exchange；和主模型相同输入、loss与训练预算。
class OneBranchLSTM(nn.Module):
    def __init__(self,feature,hidden,pcdim):
        super().__init__();self.hidden=hidden;self.pcdim=pcdim
        self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pcdim)
    def forward(self,x):
        xin=F.dropout(x,p=c.DROPOUT,training=self.training)
        z,_=self.rnn(xin)
        return self.fc(z[:,-1]).reshape(len(x),3,self.pcdim)

class TwoBranchMatchedLSTM(nn.Module):
    def __init__(self,fd,h1,h2):
        super().__init__()
        self.net1=OneBranchLSTM(2*fd['z1'].shape[1],h1,fd['pp1'].n)
        self.net2=OneBranchLSTM(2*fd['z2'].shape[1],h2,fd['pp2'].n)
    def forward(self,x1,x2):return self.net1(x1),self.net2(x2)

def lstm_formula(inputdim,pdim,h):return 4*h*(inputdim+h+2)+3*pdim*(h+1)

def matched_lstm(fd,target):
    f1=2*fd['z1'].shape[1];f2=2*fd['z2'].shape[1];p1=fd['pp1'].n;p2=fd['pp2'].n
    h1=96 # pre-fixed branch allocation, do not optimize using test results
    h2=min(range(32,768),key=lambda h:abs(lstm_formula(f1,p1,h1)+lstm_formula(f2,p2,h)-target))
    model=TwoBranchMatchedLSTM(fd,h1,h2)
    actual=nparams(model)
    if abs(actual-target)/target>.01:raise RuntimeError(f'同规模LSTM参数不匹配 {actual} vs {target}')
    return model,dict(h1=h1,h2=h2,hybrid_parameters=target,lstm_parameters=actual,relative_delta=(actual-target)/target)


In [ ]:
# 8. Freeze audit: full old checkpoint, EOF and saved forecast prediction arrays.
import io
R75_STATUS_FILE=R75/'tables/03_run_status.csv'
R76_STATUS_FILE=R76/'tables/R76_lstm_train_status.csv'
if not R75_STATUS_FILE.is_file() or not R76_STATUS_FILE.is_file():raise FileNotFoundError('缺R75/R76原运行状态表')
R75_ST=pd.read_csv(R75_STATUS_FILE);R76_ST=pd.read_csv(R76_STATUS_FILE)
assert set(zip(R75_ST.year,R75_ST.seed)) >= {(y,s) for y in c.TEST_YEARS for s in c.R75_SEEDS}
assert set(zip(R76_ST.test_year,R76_ST.seed)) >= {(y,s) for y in c.TEST_YEARS for s in c.R75_SEEDS}
R75_CFG=R75/'reports/R75_冻结实验合同.json'
if not R75_CFG.is_file():raise FileNotFoundError(R75_CFG)
parent=json.loads(R75_CFG.read_text(encoding='utf-8'))
if parent.get('dataset_sha256')!=DATASET_SHA:raise RuntimeError('R75原合同数据SHA不匹配，停止')
# Ensure model/fold-specific CID, saved EOF basis and dates are exact.
def read_frozen_model(f,fd,seed):
    yr=f['year'];folder=R75_WEIGHTS/str(yr)/str(seed) if seed in c.R75_SEEDS else WEIGHTS/str(yr)/str(seed)
    pt=folder/'best.pt';stat=folder/'status.json'
    if not pt.is_file() or not stat.is_file():raise FileNotFoundError(f'冻结权重缺失:{pt}')
    status=json.loads(stat.read_text(encoding='utf-8'))
    expected=cid_from_contract(new_contract(f,fd,seed))
    ck=torch.load(pt,map_location='cpu',weights_only=True)
    if ck.get('cid')!=expected or status.get('cid')!=expected:raise RuntimeError(f'CID不符: {yr} seed {seed}')
    model=r65_model(fd,INTERNAL_ARCH).to(DEV)
    model.load_state_dict(ck['state'],strict=True);model.eval()
    return model,sha256_file(pt),status

def audit_eof(f,fd):
    yr=f['year'];pt=R75_EOF/f'trainonly_EOF_{yr}.npz'
    if not pt.is_file():raise FileNotFoundError(pt)
    with np.load(pt,allow_pickle=False) as z:
        if str(z['dataset_sha256'].item())!=DATASET_SHA:raise RuntimeError('EOF数据集SHA不符')
        for nm,pp in [('b1',fd['pp1']),('b2',fd['pp2'])]:
            for key,value in [('mu',pp.mu),('sd',pp.sd),('V',pp.V)]:
                if not np.array_equal(z[f'{nm}_{key}'],value):raise RuntimeError(f'R75 EOF发生改变 year={yr} key={nm}_{key}')
    return sha256_file(pt)

def load_frozen_forecast(f,seed):
    p=R75_FORECAST/f'pred_{f["year"]}_seed{seed}.npz'
    if not p.is_file():raise FileNotFoundError('缺失真实原始预测数组: '+str(p))
    with np.load(p,allow_pickle=False) as z:
        if str(z['dataset_sha256'].item())!=DATASET_SHA:raise RuntimeError('R75保存预测数据集SHA不符')
        exp=np.asarray(cal[f['test']].strftime('%Y-%m-%d'),dtype='U10')
        if not np.array_equal(z['issue_date'],exp):raise RuntimeError('R75保存预测日期错配')
        if not np.array_equal(z['lead_days'],np.asarray(c.LEADS)):raise RuntimeError('R75提前期不匹配')
        return np.asarray(z['pred_b1_lon'],np.float32),np.asarray(z['pred_b2_lat'],np.float32),str(z['cid'].item())

for f in FOLDS:
    fd=make_fold_data(f);audit_eof(f,fd)
    for seed in c.R75_SEEDS:
        pred1,pred2,cid=load_frozen_forecast(f,seed)
        r=R75_ST[(R75_ST.year==f['year'])&(R75_ST.seed==seed)]
        if len(r)!=1:raise RuntimeError('状态表缺折次')
        if r.iloc[0]['cid']!=cid or cid!=cid_from_contract(new_contract(f,fd,seed)):
            raise RuntimeError('R75训练状态和预测CID不一致')
    del fd
print('[FREEZE AUDIT PASS] R75 EOF, predictions, dates, CIDs, 20 R75 runs and 20 R76 runs')


In [ ]:
# 9. HARD preflight: architecture, noise-only-input, no retune, replay original forecasts.
f=FOLDS[0];fd=make_fold_data(f);eof_sha=audit_eof(f,fd)
m,ck_sha,st=read_frozen_model(f,fd,c.R75_SEEDS[0]);old1,old2,_=load_frozen_forecast(f,c.R75_SEEDS[0]);
assert abs(nparams(m)-int(st['model_parameters']))==0
# Original saved forecast must be recoverable with identical data pipeline.
with torch.inference_mode():
    data=JointPairDataset(f['test'][:min(3,len(f['test']))],fd)
    batch=DataLoader(data,batch_size=3)
    x1,x2,_,_=next(iter(batch));q1,q2=m(x1.to(DEV),x2.to(DEV))
    r1=fd['pp1'].invert(q1.cpu().numpy());r2=fd['pp2'].invert(q2.cpu().numpy())
    maxdiff=max(float(np.max(np.abs(r1-old1[:len(r1)]))),float(np.max(np.abs(r2-old2[:len(r2)]))))
    if maxdiff>1e-4:raise RuntimeError(f'冻结模型推理无法重现 R75，最大度差={maxdiff}')
print('[PREFLIGHT PASS] max abs coordinate difference (deg)',maxdiff,'params',nparams(m),'EOF',eof_sha[:12])
# no future in 180 days observation window; confirm original fold contracts
assert all(cal[ff['train']+max(c.LEADS)].max()<cal[ff['val']].min() for ff in FOLDS)
assert all(cal[ff['val']+max(c.LEADS)].max()<cal[ff['test']].min() for ff in FOLDS)
# one random physical disturbance; targets unchanged, make both value/delta from same noisy physical stream
rng=np.random.default_rng(342)
clean1=np.stack([F1[i-H+1:i+1] for i in f['test'][:3]],axis=0)
km=np.broadcast_to(111.32*np.cos(np.deg2rad(lat1)),clean1.shape[1:])
noise=rng.normal(0,5,clean1.shape).astype(np.float32)/km[None,:,:]
z_noisy=fd['pp1'].z(clean1+noise)
assert np.isfinite(z_noisy).all() and np.array_equal(A1[f['test'][:3]+30],A1[f['test'][:3]+30].copy())
del m,fd
if DEV.type=='cuda':torch.cuda.empty_cache()


In [ ]:
# 10. Matched hardware benchmark (same GPU, batch size, precision & thread settings).
# Hardware timing does not use historic wall-time from reused 2014/2019 checkpoints (elapsed=0 != training speed).
# Training benchmark on copies, never mutates frozen best checkpoints.
f=next(t for t in FOLDS if t['year']==2021)
fd=make_fold_data(f)
prebuilt,_,_=read_frozen_model(f,fd,71)
bl,lpar=matched_lstm(fd,nparams(prebuilt))
bl=bl.to(DEV)
bench_data=JointPairDataset(f['train'][:c.BENCH_BATCH_SIZE],fd)
loader=DataLoader(bench_data,batch_size=min(c.BENCH_BATCH_SIZE,len(bench_data)))
bx1,bx2,by1,by2=next(iter(loader));bx1,bx2,by1,by2=[x.to(DEV) for x in (bx1,bx2,by1,by2)]
def sync():
    if DEV.type=='cuda':torch.cuda.synchronize()
def bench_one(model,name):
    rows=[];model.eval()
    with torch.inference_mode():
        for _ in range(c.BENCH_WARMUP):model(bx1,bx2)
        sync()
        if DEV.type=='cuda':torch.cuda.reset_peak_memory_stats()
        times=[]
        for _ in range(c.BENCH_REPEAT):
            sync();t0=time.perf_counter();model(bx1,bx2);sync();times.append(time.perf_counter()-t0)
        infer_peak=(torch.cuda.max_memory_allocated()/1024**2 if DEV.type=='cuda' else np.nan)
    rows.append(dict(model=name,step='inference',batch=len(bx1),median_ms_batch=float(np.median(times)*1000),p90_ms_batch=float(np.quantile(times,.9)*1000),throughput_samples_s=float(len(bx1)/np.median(times)),peak_cuda_allocated_mb=infer_peak))
    clone=copy.deepcopy(model).to(DEV);clone.train();opt=torch.optim.AdamW(clone.parameters(),lr=c.LR,weight_decay=c.WEIGHT_DECAY)
    use_amp=bool(DEV.type=='cuda' and c.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=use_amp)
    def step():
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=('cuda' if use_amp else 'cpu'),enabled=use_amp):
            p1,p2=clone(bx1,bx2)
            loss=.5*(r61_loss(p1.float(),by1.float(),fd,1)+r61_loss(p2.float(),by2.float(),fd,2))
        scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(clone.parameters(),1.)
        scaler.step(opt);scaler.update()
    for _ in range(c.BENCH_WARMUP):step()
    sync()
    if DEV.type=='cuda':torch.cuda.reset_peak_memory_stats()
    times=[]
    for _ in range(c.BENCH_REPEAT):
        sync();t0=time.perf_counter();step();sync();times.append(time.perf_counter()-t0)
    train_peak=(torch.cuda.max_memory_allocated()/1024**2 if DEV.type=='cuda' else np.nan)
    rows.append(dict(model=name,step='train_step',batch=len(bx1),median_ms_batch=float(np.median(times)*1000),p90_ms_batch=float(np.quantile(times,.9)*1000),throughput_samples_s=float(len(bx1)/np.median(times)),peak_cuda_allocated_mb=train_peak))
    del clone,opt,scaler
    return rows
bench_rows=bench_one(prebuilt,PAPER_NAME)+bench_one(bl,'Parameter-matched LSTM')
pd.DataFrame(bench_rows).to_csv(TABLES/'R77_hardware_microbench.csv',index=False)
training_table=R75_ST.merge(R76_ST,left_on=['year','seed'],right_on=['test_year','seed'],how='inner',validate='1:1')
training_table['train_time_comparable']=(training_table['source']=='new_rolling_training')&(training_table['elapsed_s']>0)&(training_table['elapsed_train_seconds']>0)
training_table['train_time_ratio_hybrid_to_lstm']=np.where(training_table['train_time_comparable'],training_table['elapsed_s']/training_table['elapsed_train_seconds'],np.nan)
training_table.to_csv(TABLES/'R77_R75_R76_recorded_training_times.csv',index=False)
models=[(prebuilt,PAPER_NAME,R75_WEIGHTS/'2021/71/best.pt'),(bl,'Parameter-matched LSTM',R76_WEIGHTS/'2021/71/best.pt')]
def macs_approx(model,steps):
    # Transparent analytical estimate: Conv1d, Linear, recurrent matrix multiplies only.
    # Excludes gate nonlinearities, normalization, element-wise operations and memory access.
    if isinstance(model,CrossExchangeDropGN):
        total=0
        for net in (model.net1,model.net2):
            conv=net.enc.conv; raw=net.enc.raw_dim
            total+=steps*raw*conv.in_channels*conv.out_channels*conv.kernel_size[0]
            encfc=net.enc.fc;total+=steps*encfc.in_features*encfc.out_features
            rec=net.rnn;total+=steps*4*rec.hidden_size*(rec.input_size+rec.hidden_size)
            total+=net.fc.in_features*net.fc.out_features
        total+=model.exchange1.in_features*model.exchange1.out_features
        total+=model.exchange2.in_features*model.exchange2.out_features
        return int(total)
    if isinstance(model,TwoBranchMatchedLSTM):
        return int(sum(steps*4*net.rnn.hidden_size*(net.rnn.input_size+net.rnn.hidden_size)+
                       net.fc.in_features*net.fc.out_features for net in (model.net1,model.net2)))
    raise TypeError(type(model))
complexities=[]
for model,name,pt in models:
    params=nparams(model);buffers=sum(x.numel() for x in model.buffers())
    complexities.append(dict(model=name,parameters=params,parameter_mb_fp32=params*4/1024**2,buffers_bytes=sum(x.numel()*x.element_size() for x in model.buffers()),
                             checkpoint_mb=pt.stat().st_size/1024**2 if pt.is_file() else np.nan,model_trainable_parameters=params,
                             approximate_macs_per_forecast=macs_approx(model,c.HISTORY),
                             approximate_gflops_per_forecast=2*macs_approx(model,c.HISTORY)/1e9))
pd.DataFrame(complexities).to_csv(TABLES/'R77_complexity.csv',index=False)
info={'device':str(DEV),'cuda_device':torch.cuda.get_device_name(0) if DEV.type=='cuda' else None,
      'torch_version':torch.__version__,'cuda_runtime':torch.version.cuda,'cpu':platform.processor(),
      'os':platform.platform(),'batch':len(bx1),'input_window_days':c.HISTORY,
      'reference_fold':2021,'macs_method':'analytical matrix-multiplications only, excludes activations/normalization and data movement','warmup_steps':c.BENCH_WARMUP,'timed_steps':c.BENCH_REPEAT,
      'benchmark_note':'synchronized wall-clock; torch CUDA peak allocated excludes externally owned context memory; trainer may differ due to file I/O and preprocessing'}
(REPORTS/'R77_hardware.json').write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
print('Hardware benchmark:',info);display(pd.DataFrame(bench_rows));display(pd.DataFrame(complexities))
del prebuilt,bl,fd
if DEV.type=='cuda':torch.cuda.empty_cache()


In [ ]:
# 11. Physical-coordinate noise BEFORE the frozen train-only EOF. No target noise, no test-time dropout.
# Each issue has a deterministic random stream independent of ordering or batching.
def seed_for_noise(year,seed,rep,issue):
    data=f'{c.NOISE_SEED}|{year}|{seed}|{rep}|{int(issue)}|white-independent-coordinate'
    return int.from_bytes(hashlib.blake2b(data.encode('utf8'),digest_size=8).digest(),'little')

def noise_forecast(model,fd,f,seed,noise_km,rep):
    indices=f['test'];o1=[];o2=[]
    lon_scale=111.32*np.cos(np.deg2rad(lat1))[None,:]
    for s in range(0,len(indices),c.BATCH_SIZE):
        batch=indices[s:s+c.BATCH_SIZE];x1=[];x2=[]
        for issue in batch:
            physical1=np.asarray(F1[issue-H+1:issue+1],np.float32).copy()
            physical2=np.asarray(F2[issue-H+1:issue+1],np.float32).copy()
            if noise_km>0:
                rg=np.random.default_rng(seed_for_noise(f['year'],seed,rep,issue))
                physical1+=rg.normal(0,noise_km,size=physical1.shape).astype(np.float32)/lon_scale
                physical2+=rg.normal(0,noise_km,size=physical2.shape).astype(np.float32)/111.32
            z1=fd['pp1'].z(physical1);z2=fd['pp2'].z(physical2)
            d1=np.empty_like(z1);d1[0]=0;d1[1:]=z1[1:]-z1[:-1]
            d2=np.empty_like(z2);d2[0]=0;d2[1:]=z2[1:]-z2[:-1]
            x1.append(np.concatenate([z1,d1],axis=-1));x2.append(np.concatenate([z2,d2],axis=-1))
        with torch.inference_mode():
            p1,p2=model(torch.from_numpy(np.stack(x1)).to(DEV),torch.from_numpy(np.stack(x2)).to(DEV))
        o1.append(p1.detach().float().cpu().numpy());o2.append(p2.detach().float().cpu().numpy())
    return fd['pp1'].invert(np.concatenate(o1)),fd['pp2'].invert(np.concatenate(o2))

def noise_contract(f,seed,weight_sha,eof_sha):
    obj={'protocol':'R77-noise-input-only-v1','dataset_sha256':DATASET_SHA,
         'fold':f['year'],'seed':seed,'checkpoint_sha256':weight_sha,'eof_sha256':eof_sha,
         'issues_sha':hashlib.sha256(cal[f['test']].strftime('%Y-%m-%d').str.cat(sep='|').encode()).hexdigest(),
         'noise_levels_km':list(c.NOISE_LEVELS_KM),'n_replicates':c.NOISE_REPEATS,
         'noise_seed':c.NOISE_SEED,'coordinate_noise':'physical km independent space/time, before EOF',
         'target_untouched':True,'horizon_days':list(c.LEADS)}
    return hashlib.sha256(json.dumps(obj,sort_keys=True).encode()).hexdigest(),obj

if c.ONLY_PREFLIGHT or not c.RUN_FULL:
    print('正式阶段跳过（设置 RUN_FULL=True, ONLY_PREFLIGHT=False）。')
else:
    for f_idx,f in enumerate(FOLDS,1):
        print(f'\n[{f_idx}/{len(FOLDS)}] Year {f["year"]} - frozen 2026 journal revision contract')
        fd=make_fold_data(f);eof_sha=audit_eof(f,fd)
        for seed in c.SEEDS:
            if seed in c.NEW_SEEDS:
                model,meta=run_train(f,fd,seed)
                model.eval()
            else:
                model,_,meta=read_frozen_model(f,fd,seed)
            model_sha=sha256_file((R75_WEIGHTS if seed in c.R75_SEEDS else WEIGHTS)/str(f['year'])/str(seed)/'best.pt')
            fid,_=noise_contract(f,seed,model_sha,eof_sha)
            path=TABLES/f'R77_noise_{f["year"]}_seed{seed}.csv'
            receipt=TABLES/f'R77_noise_{f["year"]}_seed{seed}.json'
            if path.exists() and receipt.exists():
                old=json.loads(receipt.read_text(encoding='utf-8'))
                try:
                    data=pd.read_csv(path)
                    n_expected=len(f['test'])*len(c.LEADS)*(1+c.NOISE_REPEATS*(len(c.NOISE_LEVELS_KM)-1))
                    if old['fid']==fid and len(data)==n_expected and (data['experiment_id']==fid).all():
                        print('[R77 VERIFIED SKIP]',f['year'],seed,len(data));del model;continue
                except Exception as err:print('[R77 INVALID CACHE - recalc]',err)
            original1=original2=None
            if seed in c.R75_SEEDS:
                original1,original2,parent_cid=load_frozen_forecast(f,seed)
            rows=[];started=time.perf_counter()
            for level in c.NOISE_LEVELS_KM:
                reps=1 if level==0 else c.NOISE_REPEATS
                for rep in range(reps):
                    if level==0 and original1 is not None:
                        p1,p2=original1,original2
                        # Also check direct network replay at first 3 samples, already passed preflight.
                    else:
                        p1,p2=noise_forecast(model,fd,f,seed,float(level),rep)
                    score=score_predictions(f,seed,p1,p2)
                    score['noise_sd_km']=float(level);score['noise_repeat']=int(rep)
                    score['experiment_id']=fid;score['checkpoint_sha256']=model_sha
                    rows.append(score)
                    print('[R77 NOISE]',f['year'],seed, 'sigma',level,'repeat',rep,'pooled RMSE',score.groupby('lead_days')['path_mse_km2'].mean().pow(.5).round(2).to_dict())
            all_noise=pd.concat(rows,ignore_index=True)
            expected=len(f['test'])*len(c.LEADS)*(1+c.NOISE_REPEATS*(len(c.NOISE_LEVELS_KM)-1))
            if len(all_noise)!=expected:raise RuntimeError('输出行数错误')
            # atomic write to avoid valid-looking files from interrupted runs
            tmp=path.with_suffix('.tmp');all_noise.to_csv(tmp,index=False);tmp.replace(path)
            receipt.write_text(json.dumps({'fid':fid,'rows':len(all_noise),'duration_s':time.perf_counter()-started,
                                           'weight_sha256':model_sha,'eof_sha256':eof_sha,'fold_year':f['year'],'seed':seed},indent=2),encoding='utf-8')
            del model,rows,all_noise
            if DEV.type=='cuda':torch.cuda.empty_cache()
        del fd;gc.collect()
    print('All 10 years x 5 seeds complete.')


In [ ]:
# 12. Stability and noise slopes, paired metrics with exact same dates.
files=sorted(TABLES.glob('R77_noise_*_seed*.csv'))
if not files:print('No formal noise files yet; preflight only')
else:
    parts=[pd.read_csv(p) for p in files];s=pd.concat(parts,ignore_index=True)
    expected={(y,z) for y in c.TEST_YEARS for z in c.SEEDS};actual=set(zip(s.test_year,s.seed))
    if c.RUN_FULL and not c.ONLY_PREFLIGHT and actual!=expected:raise RuntimeError('缺折/seed，拒绝假装完成')
    # Strict pairs: every noised forecast must match same seed/year/issue/lead clean row.
    clean=s[s.noise_sd_km==0].copy().rename(columns={'path_mse_km2':'clean_path_mse_km2','path_rmse_km':'clean_path_rmse_km'})
    keys=['test_year','seed','issue_date','target_date','lead_days']
    if clean.duplicated(keys).any():raise RuntimeError('Clean keys duplicated')
    noisy=s[s.noise_sd_km>0].merge(clean[keys+['clean_path_mse_km2','clean_path_rmse_km']],on=keys,validate='many_to_one')
    if len(noisy)!=len(s[s.noise_sd_km>0]):raise RuntimeError('有噪声样本缺少相应clean样本')
    noisy['delta_path_rmse_km']=noisy['path_rmse_km']-noisy['clean_path_rmse_km']
    # metrics are pooled within each year, seed, noise, repeat, lead, then years equally weighted.
    group=['test_year','seed','noise_sd_km','noise_repeat','lead_days']
    paired=s.groupby(group,as_index=False).agg(mse=('path_mse_km2','mean'),issue_n=('issue_date','nunique'))
    paired['year_seed_rmse_km']=np.sqrt(paired.mse)
    paired.to_csv(TABLES/'R77_per_year_seed_noise_repeat_lead.csv',index=False)
    # Repeats averaged BEFORE year/seed pooling to avoid multiplying clean counts.
    a=paired.groupby(['test_year','seed','noise_sd_km','lead_days'],as_index=False).agg(rmse=('year_seed_rmse_km','mean'))
    a.to_csv(TABLES/'R77_per_year_seed_noise_lead.csv',index=False)
    summary=a.groupby(['noise_sd_km','lead_days'],as_index=False).agg(equal_year_seed_rmse_km=('rmse','mean'),n_fold_seeds=('rmse','count'))
    base=summary[summary.noise_sd_km==0][['lead_days','equal_year_seed_rmse_km']].rename(columns={'equal_year_seed_rmse_km':'clean_rmse_km'})
    summary=summary.merge(base,on='lead_days',validate='many_to_one');summary['delta_vs_clean_km']=summary.equal_year_seed_rmse_km-summary.clean_rmse_km
    # Cluster bootstrap by year (preserve all seeds/reps for the sampled years)
    generator=np.random.default_rng(20261016)
    years=sorted(a.test_year.unique());boots=[]
    for (noise,lead),sgrp in a.groupby(['noise_sd_km','lead_days']):
        if noise==0:continue
        curr=sgrp.pivot(index='test_year',columns='seed',values='rmse')
        ref=a[(a.noise_sd_km==0)&(a.lead_days==lead)].pivot(index='test_year',columns='seed',values='rmse')
        common=curr.index.intersection(ref.index)
        d=(curr.loc[common]-ref.loc[common]).mean(axis=1).to_numpy()
        draws=generator.integers(0,len(d),size=(c.BOOT_REPS,len(d)))
        b=np.mean(d[draws],axis=1)
        boots.append({'noise_sd_km':noise,'lead_days':lead,'delta_km':np.mean(d),'CI95_lo_km':np.quantile(b,.025),'CI95_hi_km':np.quantile(b,.975),'n_years':len(d)})
    pd.DataFrame(boots).to_csv(TABLES/'R77_noise_yearblock_CI95.csv',index=False)
    summary.to_csv(TABLES/'R77_noise_summary.csv',index=False)
    # Stability per year/lead computed ACROSS five independently trained seeds
    stability=a[a.noise_sd_km==0].groupby(['test_year','lead_days'],as_index=False).agg(n_seeds=('seed','nunique'),mean_rmse=('rmse','mean'),sd_across_seeds_km=('rmse','std'),min_seed_rmse=('rmse','min'),max_seed_rmse=('rmse','max'))
    stability['cv_seed_percent']=100*stability.sd_across_seeds_km/stability.mean_rmse
    stability.to_csv(TABLES/'R77_multi_seed_stability_by_year.csv',index=False)
    allseed=a[a.noise_sd_km==0].groupby(['seed','lead_days'],as_index=False).agg(equal_year_rmse=('rmse','mean'))
    allseed.to_csv(TABLES/'R77_multi_seed_overall.csv',index=False)
    print('R7.7 noise summary:');display(summary)
    print('R7.7 stability by year:');display(stability)
    fig,ax=plt.subplots(figsize=(8,4.8))
    for lead,g in summary.groupby('lead_days'):
        ax.plot(g.noise_sd_km,g.equal_year_seed_rmse_km,'-o',label=f'+{lead} days')
    ax.set_xlabel('Input Gaussian noise SD (km)');ax.set_ylabel('Equal-year & equal-seed RMSE (km)');ax.set_title('hybrid deep-learning prediction — input noise robustness');ax.legend();ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(FIG/'R77_Fig01_Noise_Robustness.png',dpi=c.FIG_DPI);fig.savefig(FIG/'R77_Fig01_Noise_Robustness.pdf');plt.close(fig)
    fig,ax=plt.subplots(figsize=(10,4.8))
    for lead,g in stability.groupby('lead_days'):
        ax.plot(g.test_year,g.sd_across_seeds_km,'-o',label=f'+{lead} days')
    ax.set_xlabel('Test year');ax.set_ylabel('SD across 5 seeds (km)');ax.set_title('hybrid deep-learning prediction — seed sensitivity');ax.legend();ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(FIG/'R77_Fig02_Seed_Stability.png',dpi=c.FIG_DPI);fig.savefig(FIG/'R77_Fig02_Seed_Stability.pdf');plt.close(fig)
    timepair=training_table[training_table.train_time_comparable]
    print('Reused legacy 2014/2019 excluded from training wall-time comparison. Eligible year/seed:',len(timepair))
    print('Historic training duration median (s):',timepair[['elapsed_s','elapsed_train_seconds']].median().to_dict())


In [ ]:
# 13. Archive results, provenance, caveats; do not package multi-GB weights.
arch=ROOT/'R77_上传给ChatGPT_复杂度噪声鲁棒性多Seed稳定性.zip'
receipts=sorted(TABLES.glob('R77_noise_*_seed*.json'))
complete=len(receipts)==len(c.TEST_YEARS)*len(c.SEEDS) and (TABLES/'R77_noise_summary.csv').exists()
manifest={'model':PAPER_NAME,'status':'complete' if complete else 'partial','expected_run_pairs':len(c.TEST_YEARS)*len(c.SEEDS),
          'completed_run_pairs':len(receipts),'dataset_sha256':DATASET_SHA,'test_years':list(c.TEST_YEARS),
          'all_seeds':list(c.SEEDS),'noised_input_only':True,'checkpoint_existing_seeds':list(c.R75_SEEDS),
          'trained_new_seeds':list(c.NEW_SEEDS),'outcome_not_yet_independent_blind':True,
          'training_time_note':'Original R75 reused-legacy checkpoint reports 0 training seconds; exclude those entries from comparable training costs.',
          'quantities':'clean/historical noise evaluated against GLORYS path ground truth; do not combine with ADT validation metrics',
          'clock_utc':datetime.now(timezone.utc).isoformat()}
(REPORTS/'R77_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
files_to_ship=[]
for d in [TABLES,FIG,REPORTS]:files_to_ship.extend([p for p in d.rglob('*') if p.is_file()])
for sfile in WEIGHTS.glob('*/*/status.json'):files_to_ship.append(sfile)
for hfile in WEIGHTS.glob('*/*/history.csv'):files_to_ship.append(hfile)
if c.INCLUDE_WEIGHTS_IN_ZIP:
    files_to_ship.extend(WEIGHTS.glob('*/*/best.pt'))
with zipfile.ZipFile(arch,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7,allowZip64=True) as z:
    for q in sorted(set(files_to_ship)):z.write(q,arcname=str(q.relative_to(RUN)))
with zipfile.ZipFile(arch) as z:
    if z.testzip() is not None:raise RuntimeError('ZIP CRC失败')
checksum=sha256_file(arch);sha=arch.with_suffix(arch.suffix+'.sha256')
sha.write_text(f'{checksum}  {arch.name}\n',encoding='utf-8')
print('Saved',arch,arch.stat().st_size,'bytes, SHA256=',checksum,'manifest',manifest['status'])
print('Submit both ZIP and .sha256. Train checkpoints remain on local D: drive.')
